In [ ]:
import sys
import itertools
sys.path.append('../')
import random
from core import LADTransferTreeBoost, LSTransferTreeBoost
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb #baseline
from utils import *
from baselines import *
from friedman1 import *
import friedman_config as c

In [ ]:
#also run mlp finetuning
ablation_transfer_normal_normal = pd.DataFrame(columns = ['seed', 'target_instances', 'd', 'method', 'base_lr', 'fine_tuning_lr', 'dropout_rate', 'batch_norm',
                                                          'val_rmse', 'val_mae', 'rmse', 'mae'])

for seed in c.seed_list:
    for target_instances in c.target_instances_list:
    
        X_target_test, y_target_test = friedman1(n_samples=c.test_size, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #do NOT add noise to test set!!!!
        X_target_val, y_target_val = friedman1(n_samples=c.val_size, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed + 10) #do NOT add noise to test set!!!!
        X_target_train, y_target_train = friedman1(n_samples=target_instances, add_noise = True, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #add noise to train set
        for d in c.d_list:
            X_source_train, y_source_train = friedman1_altered(n_samples=1000, add_noise = True, noise_distribution = 'gaussian',
                                                            n_features=10, d=d, shift_seed=seed, random_seed = seed) #also add noise to source (only train here)


            for base_lr in c.base_lrs:
                for finetuning_lr in c.fine_tuning_lrs:
                    for dropout_rate in c.dropout_list:
                        for batch_norm in c.include_batch_norm:

                            method = f'MLP'
                            mlp = MLP(10, 100, 100, 100, 1, dropout_rate=dropout_rate, include_batch_norm=batch_norm)
                            dataloader_train = process_dataset_for_base_network(X_source_train, y_source_train)
                            mlp, train_loss, val_loss = train_mlp_on_source(dataloader_train, mlp, epochs=1000)
                            dataloader_train, dataloader_val, dataloader_test = process_datasets_for_finetuning(X_target_train, y_target_train,
                                                        X_target_val, y_target_val, X_target_test, y_target_test, batch_size=32)
                            
                            mlp, train_loss, val_loss = finetune_mlp_on_target(dataloader_train, dataloader_val, mlp, epochs=1000, freeze_layers=None)
                            val_rmse, val_mae = test_final_mlp(dataloader_val, mlp)
                            rmse, mae = test_final_mlp(dataloader_test, mlp)
                            ablation_transfer_normal_normal.loc[len(ablation_transfer_normal_normal)] = [seed, target_instances, d, method, base_lr, finetuning_lr, dropout_rate, batch_norm, val_rmse, 
                                                                                                         val_mae, rmse, mae]
                            ablation_transfer_normal_normal.to_csv(f'results/MLP_ablation_friedman.csv')





                            
        

